# Multiclass Chain Model Evaluator

Evaluate the saved random-forest chain-selection baseline on the RL v4 test set. Run `best_chain_model.py` first.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'dbs' / 'combined_master.csv').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not locate project root.')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from machine_learning.RL.two_opt_chain_rl_v4.data_processing import prepare_datasets
from machine_learning.supervised_learning.multiclass.best_chain_model import (
    add_utility_and_targets,
    build_candidate_frame,
    chain_order,
)

sns.set_theme(style='whitegrid', context='notebook')
RUN_DIR = PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed42'
RUN_DIR2 = PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed43'
RUN_DIR3 = PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed44'
RUN_DIRS = [
    RUN_DIR,
    RUN_DIR2,
    RUN_DIR3
]
SAVE_FIGURES = True
RUN_DIRS = [Path(path) for path in RUN_DIRS]
if not RUN_DIRS:
    raise ValueError('Provide at least one run directory in RUN_DIRS.')
EVALUATION_LABEL = RUN_DIRS[0].name if len(RUN_DIRS) == 1 else f'aggregate_{len(RUN_DIRS)}_runs'
FIGURE_ROOT = PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'evaluator_figures'
FIGURE_DIR = FIGURE_ROOT / EVALUATION_LABEL
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('Run directories:')
for run_dir in RUN_DIRS:
    print(f'  - {run_dir}')

## Load Run

In [ ]:
run_records = []
for run_dir in RUN_DIRS:
    required_files = [run_dir / 'model.joblib', run_dir / 'test_predictions.csv', run_dir / 'run_config.json']
    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Missing evaluator artifacts for {run_dir}: {missing}')
    model = joblib.load(run_dir / 'model.joblib')
    predictions = pd.read_csv(run_dir / 'test_predictions.csv')
    run_config = json.loads((run_dir / 'run_config.json').read_text())
    run_records.append({
        'run_dir': run_dir,
        'run_name': run_dir.name,
        'model': model,
        'test_predictions': predictions,
        'run_config': run_config,
    })

reference_config = run_records[0]['run_config']
TWO_QUBIT_WEIGHT = float(reference_config['two_qubit_weight'])
DEPTH_WEIGHT = float(reference_config['depth_weight'])
RUNTIME_WEIGHT = float(reference_config['runtime_weight'])
for record in run_records[1:]:
    config = record['run_config']
    if (
        float(config['two_qubit_weight']) != TWO_QUBIT_WEIGHT
        or float(config['depth_weight']) != DEPTH_WEIGHT
        or float(config['runtime_weight']) != RUNTIME_WEIGHT
    ):
        raise ValueError('All selected runs must use the same reward weights.')

for record in run_records:
    record['test_predictions'] = record['test_predictions'].copy()
    record['test_predictions']['run_name'] = record['run_name']

test_predictions = pd.concat([record['test_predictions'] for record in run_records], ignore_index=True)
per_run_summary = pd.DataFrame([
    {
        'run_name': record['run_name'],
        'Mean utility': record['test_predictions']['utility'].mean(),
        'Mean 2Q reduction (%)': record['test_predictions']['total_two_qubit_gate_reduction_pct'].mean(),
        'Mean depth reduction (%)': record['test_predictions']['total_depth_reduction_pct'].mean(),
        'Mean runtime (s)': record['test_predictions']['total_runtime_seconds'].mean(),
        'Mean runtime score': record['test_predictions']['runtime_score'].mean(),
        'Classification accuracy': record['test_predictions']['correct'].mean(),
    }
    for record in run_records
])
print(f"Runs loaded: {len(run_records)}")
print(f"Model: {type(run_records[0]['model']).__name__}")
print(f"Test circuits per run: {len(run_records[0]['test_predictions'])}")
print(f'Weights: 2Q={TWO_QUBIT_WEIGHT}, depth={DEPTH_WEIGHT}, runtime={RUNTIME_WEIGHT}')
display(per_run_summary)
display(run_records[0]['test_predictions'].head())

## Test Performance

In [ ]:
summary = pd.DataFrame({
    'Mean': per_run_summary.drop(columns=['run_name']).mean(),
    'Std': per_run_summary.drop(columns=['run_name']).std(ddof=0),
}).fillna(0.0)
display(summary.style.format('{:.4f}'))

metric_plot = summary.loc[['Mean utility', 'Mean 2Q reduction (%)', 'Mean depth reduction (%)', 'Mean runtime (s)'], ['Mean']]
ax = metric_plot.plot.bar(legend=False, figsize=(8, 4), color='#2f6690')
ax.set_ylabel('Average value across runs')
ax.set_title('Multiclass Baseline Average Test Metrics')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(FIGURE_DIR / 'test_metrics.png', dpi=180)
plt.show()

## Selected Chains

In [ ]:
chain_counts = (
    test_predictions.groupby(['run_name', 'predicted_chain']).size().rename('circuits').reset_index()
    .groupby('predicted_chain')['circuits'].mean().sort_values(ascending=False)
    .rename_axis('chain_name').reset_index(name='avg_circuits')
)
display(chain_counts)
plot_counts = chain_counts.sort_values('avg_circuits', ascending=True)
ax = plot_counts.plot.barh(x='chain_name', y='avg_circuits', legend=False, figsize=(9, 7), color='#3c6e71')
ax.set_xlabel('Average test circuits selected per run')
ax.set_ylabel('')
ax.set_title('Multiclass Baseline Average Chain Selection')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(FIGURE_DIR / 'selected_chain_counts.png', dpi=180)
plt.show()

## Baseline Comparison

The per-circuit oracle is descriptive only. It selects the highest-utility measured candidate for each test circuit.

In [ ]:
bundle = prepare_datasets(save=False, feature_batch='plus_all_features')
test_frame = bundle['test_df']
class_order = chain_order(bundle['optimizer_names'])
candidates = build_candidate_frame(test_frame, bundle['optimizer_names'], bundle['runtime_denominator'])
candidates, targets = add_utility_and_targets(candidates, TWO_QUBIT_WEIGHT, DEPTH_WEIGHT, RUNTIME_WEIGHT, class_order)

lookup = candidates.set_index(['original_circuit', 'chain_name'])
model_rows = lookup.loc[list(zip(test_predictions['original_circuit'], test_predictions['predicted_chain']))].reset_index()
original_rows = candidates[candidates['chain_name'] == '__original__']
fixed_scores = candidates[candidates['chain_name'] != '__original__'].groupby('chain_name')['utility'].mean()
best_fixed_chain = fixed_scores.idxmax()
fixed_rows = candidates[candidates['chain_name'] == best_fixed_chain]
oracle_rows = lookup.loc[list(zip(targets.index, targets.values))].reset_index()

def method_summary(rows, method):
    return {
        'method': method,
        'utility': rows['utility'].mean(),
        '2Q reduction (%)': rows['total_two_qubit_gate_reduction_pct'].mean(),
        'depth reduction (%)': rows['total_depth_reduction_pct'].mean(),
        'runtime (s)': rows['total_runtime_seconds'].mean(),
    }

model_label = 'Multiclass random forest' if len(run_records) == 1 else f'Multiclass random forest ({len(run_records)}-run average)'
comparison = pd.DataFrame([
    method_summary(model_rows, model_label),
    method_summary(original_rows, 'No optimization'),
    method_summary(fixed_rows, f'Best fixed chain: {best_fixed_chain}'),
    method_summary(oracle_rows, 'Per-circuit oracle'),
]).set_index('method')
display(comparison.style.format('{:.4f}'))
print(f'Best fixed chain: {best_fixed_chain}')

In [ ]:
axes = comparison[['2Q reduction (%)', 'depth reduction (%)', 'runtime (s)']].plot.bar(subplots=True, figsize=(12, 8), legend=False)
for axis in np.atleast_1d(axes):
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=35)
plt.suptitle('Test-Set Baseline Comparison')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(FIGURE_DIR / 'test_baseline_comparison.png', dpi=180)
plt.show()

## Per-Circuit Utility Regret

In [ ]:
per_circuit = model_rows.set_index('original_circuit')[['chain_name', 'utility']].rename(columns={'chain_name': 'predicted_chain', 'utility': 'model_utility'})
per_circuit['oracle_utility'] = oracle_rows.set_index('original_circuit')['utility']
per_circuit['regret'] = per_circuit['oracle_utility'] - per_circuit['model_utility']
display(per_circuit.sort_values('regret', ascending=False).head(15))

ax = per_circuit['regret'].sort_values(ascending=False).reset_index(drop=True).plot(figsize=(9, 4), color='#d1495b')
ax.set_xlabel('Test circuits sorted by regret')
ax.set_ylabel('Oracle utility - model utility')
ax.set_title('Per-Circuit Utility Regret')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(FIGURE_DIR / 'per_circuit_utility_regret.png', dpi=180)
plt.show()